# omicsTL - Transfer Learning Example

<div>
<img src="images/tl_arch1.png" width="500" style="background-color: white;"/>
</div>

This notebook demonstrates the use of omicsTL to create a transfer learning model for synthetic data generated using real data objects.

We will start by loading the required functions.



In [3]:
import warnings
warnings.filterwarnings("always")

from omicstl.simulation_utils.data_generation import response_function, generate_synth_data

The first step is to generate two synthetic data frames. We will be using two different viral datasets with aligned features as the real data sources for the simulated data. The first will represent the source data that we are training the "base" model on, and the second will represent the target data.

In [4]:
import pandas as pd

source_real_data_path = "data/source_data_real.csv"
target_real_data_path = "data/target_data_real.csv"

source_real_data = pd.read_csv(source_real_data_path, index_col=0)
target_real_data = pd.read_csv(target_real_data_path, index_col=0)

# Non-trivial response function
response_fn = response_function("tanh(df[, 2]) + df[, 1] * df[, ncol(df)] ^ 2")

#TODO: categorical response

num_features = 100
num_samples_source = 100
num_samples_target = 25
samples_target_test = 50

source_synth_data, _, _ = generate_synth_data(
    data = source_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_source, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 # signal to noise ratio
)

target_synth_data, _, _ = generate_synth_data(
    data = target_real_data, # input data
    num_features = num_features, # number of output features
    num_samples = num_samples_target + samples_target_test, # number of output samples
    response_fn = response_fn, # response function
    snr = 1 # signal to noise ratio
)

display(source_synth_data)
display(target_synth_data)

R[write to console]: also installing the dependencies ‘lazyeval’, ‘parallelly’, ‘codetools’, ‘lobstr’, ‘rex’, ‘future’, ‘globals’, ‘carrier’, ‘covr’, ‘listenv’


R[write to console]: also installing the dependencies ‘audio’, ‘iterators’, ‘beepr’, ‘pbmcapply’, ‘foreach’, ‘doFuture’, ‘future.apply’, ‘ntfy’, ‘RPushbullet’


R[write to console]: Error in FUN(X[[i]], ...) : there is no package called ‘viRandomForests’



RRuntimeError: Error in FUN(X[[i]], ...) : there is no package called ‘viRandomForests’


In [ ]:
#Set response column name

source_synth_data.rename(columns={source_synth_data.columns[0]: 'response'}, inplace=True)
target_synth_data.rename(columns={target_synth_data.columns[0]: 'response'}, inplace=True)

Then we can split the target data into train and test sets.

In [ ]:
from sklearn.model_selection import train_test_split

target_synth_train, target_synth_test = train_test_split(
    target_synth_data,
    train_size = num_samples_target,
    test_size = samples_target_test
)

The data is now ready to be used for transfer learning.

We provide a DatasetContainer helper class to keep the different datasets organized. Optionally, id_tuple can also be set which is used to label the scenario and replicate IDs if performing large scale simulation studies. These IDs are passed to the output after model fitting.

In [ ]:
from omicstl.simulation_utils.data_utils import DatasetContainer
datasets = DatasetContainer(
	source_data=source_synth_data,
	target_data=target_synth_train,
	target_test_data=[target_synth_test]
)
datasets.set_response_column("response") # Identify response column

We enable automatic tuning over a parameter grid for both deep learning models:

In [ ]:
param_grid = {
	"dropout": [0.25, 0.5],
	"n_latent_dims": [2],
	"hidden_dim_base": [6],
	"lr": [0.01, 0.001],
	"source_epochs": [1000],
	"target_epochs": [1000],
	"freeze": ["none"],
	"weight_decay": [1e-4, 1e-2],
	"gamma": [1, 2, 3]
}

Deep learning models can be fit using a DatasetContainer and parameter grid using fit_dl_model, which returns a dataframe with results, the trained transfer learning model, and the model trained on only the target dataset.

In [ ]:
import torch
from torch import device
import random
from omicstl.simulation_utils.model_utils import fit_dl_model

random.seed(42)
torch.manual_seed(42)
out, model, model_targetonly = fit_dl_model(
	datasets,
	"mult_vae",
	device("cpu"),
	param_grid
)
out

Best parameter combination found. Metric rmse with value 2.157968437600914
{'dropout': np.float64(0.25), 'n_latent_dims': np.float64(2.0), 'hidden_dim_base': np.float64(6.0), 'lr': np.float64(0.01), 'source_epochs': np.float64(1000.0), 'target_epochs': np.float64(1000.0), 'freeze': 'none', 'weight_decay': np.float64(0.0001), 'gamma': np.float64(1.0)}
0     2.157968
1     2.224364
2     2.266779
3     2.245789
4     2.229974
5     2.212272
6     2.290503
7     2.273986
8     2.275661
9     2.202477
10    2.266442
11    2.276711
12    2.220428
13    2.234893
14    2.333948
15    2.360864
16    2.218313
17    2.210387
18    2.276812
19    2.292806
20    2.218567
21    2.239564
22    2.408286
23    2.368721
Name: rmse, dtype: float64
Best parameter combination found. Metric rmse with value 0.5982478590637388
{'dropout': np.float64(0.25), 'n_latent_dims': np.float64(2.0), 'hidden_dim_base': np.float64(6.0), 'lr': np.float64(0.01), 'source_epochs': np.float64(1000.0), 'target_epochs': np.flo

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:952: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_vae,mult_vae,target,0.823461,0.654784,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0001,1.0
1,None,None,test_0,mult_vae,mult_vae,target_nosource,0.842927,0.635367,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.0100,2.0


In [ ]:
out, model, model_targetonly = fit_dl_model(
	datasets,
	"mult_mlp",
	device("cpu"),
	param_grid
)
out

Best parameter combination found. Metric rmse with value 2.0256740104178994
{'dropout': np.float64(0.5), 'n_latent_dims': np.float64(2.0), 'hidden_dim_base': np.float64(6.0), 'lr': np.float64(0.01), 'source_epochs': np.float64(1000.0), 'target_epochs': np.float64(1000.0), 'freeze': 'none', 'weight_decay': np.float64(0.01), 'gamma': np.float64(2.0)}
0     2.161144
1     2.216860
2     2.369179
3     2.299927
4     2.215471
5     2.191920
6     2.275989
7     2.334533
8     2.079647
9     2.205880
10    2.304380
11    2.245730
12    2.045631
13    2.112882
14    2.265192
15    2.213266
16    2.138111
17    2.025674
18    2.315349
19    2.231530
20    2.036634
21    2.056536
22    2.319462
23    2.239392
Name: rmse, dtype: float64
Best parameter combination found. Metric rmse with value 0.5551401235768127
{'dropout': np.float64(0.25), 'n_latent_dims': np.float64(2.0), 'hidden_dim_base': np.float64(6.0), 'lr': np.float64(0.01), 'source_epochs': np.float64(1000.0), 'target_epochs': np.float

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:952: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, test_results], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,...,dropout,hidden_dim_base,n_latent_dims,source_epochs,target_epochs,freeze,z_dim_base,lr,weight_decay,gamma
0,None,None,test_0,mult_mlp,mult_mlp,target,0.880082,0.678119,NaN,NaN,...,0.50,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.01,2.0
1,None,None,test_0,mult_mlp,mult_mlp,target_nosource,0.841477,0.655694,NaN,NaN,...,0.25,6.0,2.0,1000.0,1000.0,none,12.0,0.01,0.01,2.0


The random forest based models can also be fit using a python interface to the R modeling code.

In [ ]:
from omicstl.transfer_forest import load_r_functions
from omicstl.simulation_utils.model_utils import fit_rf_model

load_r_functions()

random.seed(42)

out, model = fit_rf_model(datasets)
out

/opt/venv/lib/python3.12/site-packages/omicstl/simulation_utils/model_utils.py:1069: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  results = pd.concat([results, pd.DataFrame([test_row])], ignore_index=True)


,scenario,replicate,split,model,model_type,model_id,rmse,mae,acc,f1,mcc,precision,recall
0,None,None,test_0,rf,pred_0,target,0.840996,0.642351,NaN,NaN,NaN,NaN,NaN
1,None,None,test_0,rf,pred_1,target,0.808818,0.603667,NaN,NaN,NaN,NaN,NaN
2,None,None,test_0,rf,pred_2,target,1.826970,1.615897,NaN,NaN,NaN,NaN,NaN
3,None,None,test_0,rf,pred_3,target,0.828257,0.625472,NaN,NaN,NaN,NaN,NaN
4,None,None,test_0,rf,pred_ensemble,target,0.365294,0.262149,NaN,NaN,NaN,NaN,NaN


Advanced users can also work directly with the base classes we provide for each TL model via TransferForest() and MultiViewModel(). 